# Retail Sales ETL using PySpark

## 1. Data Loading

In [ ]:
df = spark.read.table("retail_store_sales")
df.show()

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|Transaction ID|Customer ID|            Category|        Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|   TXN_6867343|    CUST_09|          Patisserie| Item_10_PAT|          18.5|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            true|
|   TXN_3731986|    CUST_22|       Milk Products|Item_17_MILK|          29.0|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            true|
|   TXN_9303719|    CUST_02|            Butchers| Item_12_BUT|          21.5|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           false|
|   TXN_9458126|    CUST_06|           Beverages| Item_16_

## 2. Data Exploration

In [ ]:
#Getting the total number of rows and columns in the DataFrame
total_rows = df.count()
total_cols = len(df.columns)

# Print the results
print(f"Total Rows: {total_rows}")
print(f"Total Columns: {total_cols}")

Total Rows: 12575
Total Columns: 12


In [ ]:
df.printSchema()

root
 |-- Transaction ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Price Per Unit: double (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Total Spent: double (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: date (nullable = true)
 |-- Discount Applied: boolean (nullable = true)



In [ ]:
#Checking for null
from pyspark.sql.functions import col, sum
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+
|Transaction ID|Customer ID|Category|Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|
+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+
|             0|          0|       0|1213|           609|     604|        604|             0|       0|               0|            4199|
+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+



## 3. Data Cleaning

In [4]:
from pyspark.sql.functions import col, sum

#Removing NaN values
df = df.dropna(subset=["Price Per Unit", "Quantity"])

In [7]:
#filter the invalid numerical
df = df.filter(
    (col("Price Per Unit") > 0) &
    (col("Quantity") > 0)
)

## 4. Data Transformation

In [6]:
#Coverting to date format
from pyspark.sql.functions import to_date, col

df = df.withColumn(
    "Transaction Date",
    to_date(col("Transaction Date"), "dd-MM-yyyy")
)

In [8]:
#filling categorical columns
from pyspark.sql.functions import col, sum

df = df.fillna({
    "Item":"Unknown"
})


In [9]:
#filtering invaild data
df = df.filter(col("Item") != "Unknown")

In [10]:
# filling boolean columns
df = df.fillna({"Discount Applied":False})

In [11]:
from pyspark.sql.functions import col, sum
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+
|Transaction ID|Customer ID|Category|Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|
+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+
|             0|          0|       0|   0|             0|       0|          0|             0|       0|               0|               0|
+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+



## 5. Feature Engineering

In [12]:
#recalculating the total spent for better clarification

df = df.withColumn("Total_Sales",col("Quantity")*col("Price Per Unit"))

In [13]:
#Validating the Total spent column
df = df.withColumn(
    "Mismatch_Flag",
    col("Total Spent") != col("Total_Sales")
)

In [14]:
from pyspark.sql.functions import year, month

#Adding the year and month columns
df = df.withColumn("Year", year(col("Transaction Date"))) \
       .withColumn("Month", month(col("Transaction Date")))

In [15]:
#check for duplicates
total_rows = df.count()
unique_rows = df.distinct().count()

print("Total rows: ", total_rows)
print("unique rows: ", unique_rows)


Total rows:  11362
unique rows:  11362


In [16]:
#converting the datatype
df = df.withColumn("Quantity",col("Quantity").cast("int"))

In [17]:
df.printSchema()

root
 |-- Transaction ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = false)
 |-- Price Per Unit: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Total Spent: double (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: date (nullable = true)
 |-- Discount Applied: boolean (nullable = false)
 |-- Total_Sales: double (nullable = true)
 |-- Mismatch_Flag: boolean (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)



In [18]:
#checking for negative values
df.filter(col("Quantity")< 0).show()


+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+-----------+-------------+----+-----+
|Transaction ID|Customer ID|Category|Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|Total_Sales|Mismatch_Flag|Year|Month|
+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+-----------+-------------+----+-----+
+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+-----------+-------------+----+-----+



In [19]:
#checking for negative values
df.filter(col("Price Per Unit")<0)

DataFrame[Transaction ID: string, Customer ID: string, Category: string, Item: string, Price Per Unit: double, Quantity: int, Total Spent: double, Payment Method: string, Location: string, Transaction Date: date, Discount Applied: boolean, Total_Sales: double, Mismatch_Flag: boolean, Year: int, Month: int]

In [20]:
df.show()

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+-----------+-------------+----+-----+
|Transaction ID|Customer ID|            Category|        Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|Total_Sales|Mismatch_Flag|Year|Month|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+-----------+-------------+----+-----+
|   TXN_6867343|    CUST_09|          Patisserie| Item_10_PAT|          18.5|      10|      185.0|Digital Wallet|  Online|      2024-04-08|            true|      185.0|        false|2024|    4|
|   TXN_3731986|    CUST_22|       Milk Products|Item_17_MILK|          29.0|       9|      261.0|Digital Wallet|  Online|      2023-07-23|            true|      261.0|        false|2023|    7|
|   TXN_9303719|    CUST_02|  

In [21]:
#Total Sales by Category
df.groupby("Category")\
    .sum("Total_Sales")\
    .orderBy("sum(Total_Sales)", ascending=False)\
    .show()

+--------------------+----------------+
|            Category|sum(Total_Sales)|
+--------------------+----------------+
|            Butchers|        197426.0|
|Electric househol...|        192441.5|
|           Beverages|        187978.5|
|           Furniture|        186527.0|
|                Food|        184645.0|
|Computers and ele...|        180902.5|
|          Patisserie|        172330.5|
|       Milk Products|        170747.5|
+--------------------+----------------+



In [23]:
#Total Sales by Payment Method
df.groupBy("Payment Method")\
    .sum("Total_Sales")\
    .orderBy("sum(Total_Sales)",ascending=False)\
    .show()

+--------------+----------------+
|Payment Method|sum(Total_Sales)|
+--------------+----------------+
|          Cash|        513676.0|
|   Credit Card|        481135.0|
|Digital Wallet|        478187.5|
+--------------+----------------+



In [24]:
#Sales by Location
df.groupBy("Location")\
    .sum("Total_Sales")\
    .orderBy("sum(Total_Sales)",ascending=False)\
    .show()

+--------+----------------+
|Location|sum(Total_Sales)|
+--------+----------------+
|  Online|        749431.0|
|In-store|        723567.5|
+--------+----------------+



In [25]:
#Discount Impact
df.groupBy("Discount Applied") \
  .sum("Total_Sales") \
  .show()

+----------------+----------------+
|Discount Applied|sum(Total_Sales)|
+----------------+----------------+
|            true|        496880.5|
|           false|        976118.0|
+----------------+----------------+



In [26]:
#Top Selling Items
df.groupBy("Item") \
  .sum("Total_Sales") \
  .orderBy("sum(Total_Sales)", ascending=False) \
  .show(5)

+------------+----------------+
|        Item|sum(Total_Sales)|
+------------+----------------+
| Item_25_FUR|         25256.0|
| Item_25_EHE|         23083.0|
| Item_25_BUT|         21894.0|
| Item_24_FUR|         21172.0|
|Item_25_FOOD|         20541.0|
+------------+----------------+
only showing top 5 rows


In [27]:
#Now your data behaves like a SQL table
df.createOrReplaceTempView("sales")

In [28]:
#Category-wise Sales
spark.sql("""
SELECT Category, SUM(Total_Sales) AS Total_Sales
FROM sales
GROUP BY Category
ORDER BY Total_Sales DESC
""").show()

+--------------------+-----------+
|            Category|Total_Sales|
+--------------------+-----------+
|            Butchers|   197426.0|
|Electric househol...|   192441.5|
|           Beverages|   187978.5|
|           Furniture|   186527.0|
|                Food|   184645.0|
|Computers and ele...|   180902.5|
|          Patisserie|   172330.5|
|       Milk Products|   170747.5|
+--------------------+-----------+



In [29]:
#Payment Method Analysis
spark.sql("""
SELECT `Payment Method`, SUM(Total_Sales) AS Total
FROM sales
GROUP BY `Payment Method`
""").show()

+--------------+--------+
|Payment Method|   Total|
+--------------+--------+
|   Credit Card|481135.0|
|Digital Wallet|478187.5|
|          Cash|513676.0|
+--------------+--------+



In [30]:
#Location-wise Sales
spark.sql("""
SELECT Location, SUM(Total_Sales) AS Total
FROM sales
GROUP BY Location
ORDER BY Total DESC
""").show()

+--------+--------+
|Location|   Total|
+--------+--------+
|  Online|749431.0|
|In-store|723567.5|
+--------+--------+



In [31]:
#Discount Impact
spark.sql("""
SELECT `Discount Applied`, SUM(Total_Sales) AS Total
FROM sales
GROUP BY `Discount Applied`
""").show()

+----------------+--------+
|Discount Applied|   Total|
+----------------+--------+
|            true|496880.5|
|           false|976118.0|
+----------------+--------+



In [32]:
df.show(100)

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+-----------+-------------+----+-----+
|Transaction ID|Customer ID|            Category|        Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|Total_Sales|Mismatch_Flag|Year|Month|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+-----------+-------------+----+-----+
|   TXN_6867343|    CUST_09|          Patisserie| Item_10_PAT|          18.5|      10|      185.0|Digital Wallet|  Online|      2024-04-08|            true|      185.0|        false|2024|    4|
|   TXN_3731986|    CUST_22|       Milk Products|Item_17_MILK|          29.0|       9|      261.0|Digital Wallet|  Online|      2023-07-23|            true|      261.0|        false|2023|    7|
|   TXN_9303719|    CUST_02|  

In [33]:
#Table View in Databricks
display(df)

DataFrame[Transaction ID: string, Customer ID: string, Category: string, Item: string, Price Per Unit: double, Quantity: int, Total Spent: double, Payment Method: string, Location: string, Transaction Date: date, Discount Applied: boolean, Total_Sales: double, Mismatch_Flag: boolean, Year: int, Month: int]

## 6. Data Export

In [ ]:
#Create Volume
spark.sql("""
CREATE VOLUME workspace.default.my_volume
""")

DataFrame[]

In [ ]:
# save csv
df.coalesce(1).write.mode("overwrite").csv(
    "/Volumes/workspace/default/my_volume/sales_cleaned",
    header=True
)